# Meeting 13: solvers and sparse matrices

**CS 301, Introduction to Data Science, NJIT.** Unit II, meeting 13, the second notebook.

The first notebook of the meeting solved systems of two and three equations. Real systems have thousands or millions, and this notebook is about solving them: the method of elimination written with loops, so that its cost can be counted; the same system solved by inverting the matrix and by `np.linalg.solve`, timed; what round-off does to a solution, and how the condition number predicts it; and matrices that are mostly zeros, stored without the zeros, for which solving and multiplying cost a fraction of what the full matrix would.

Do it **before meeting 13**, after the first notebook of the meeting, and upload it with the first one.

## How this works

- **This notebook needs no GPU.** Leave its runtime as it is.
- **First** choose **Save a copy in Drive** in the **File** menu, so that your changes are kept, and run the cells in order, from the top. To run a cell, click it and press **Shift + Enter**.
- **You may work with your group.** **Every student uploads their own copy.**
- **Six parts**, with code to run. Five places ask you to write a line, a number read from a table, or a function that an AI assistant writes, marked <font color="#c00000"><b>Your turn</b></font>, each with a **check** whose message says what to look at. Two cells take up to a minute to run.
- **Four cells ask for your own words**: one marked <font color="#c00000"><b>Your prompt</b></font>, the prompt you give an AI assistant, and three marked <font color="#c00000"><b>Your thinking</b></font>. **They are what this notebook is graded on**, with the four cells of the first notebook. Write them yourself, even when your group worked together.
- **One post** on the discussion board of meeting 13, **before the meeting begins**, after Part 6, as a reply to your group's first post. One member may post for the group; **every post begins with the names of all the members of the group**, and earns the attention mark of this meeting for everyone it names.
- At the end, upload **both notebooks** to Canvas together, as the last section says.

<font color="#c00000" size="5"><b>Your turn.</b></font> Double-click this cell and replace this line with your name and the names of the other members of your group.

In [ ]:
# Setup. Run this cell first.
import ast
import inspect
import textwrap
import time

import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sparse
import scipy.sparse.linalg as sparse_linalg

rng = np.random.default_rng(0)           # a source of random numbers


def best_time(f, repeats=3):
    """The shortest of a few runs of f(), in seconds: the run least disturbed by anything else on the machine."""
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        f()
        times.append(time.perf_counter() - start)
    return min(times)


def in_words(number):
    """A large number in words: 6.4 million, 1.2 billion, 3.5 trillion."""
    for size, word in ((1e12, "trillion"), (1e9, "billion"), (1e6, "million"), (1e3, "thousand")):
        if number >= size:
            return f"{number / size:.3g} {word}"
    return f"{number:.3g}"


def duration(seconds):
    """A time in the unit that suits it."""
    if seconds < 1:
        return f"{seconds * 1000:.3g} milliseconds"
    if seconds < 120:
        return f"{seconds:.3g} seconds"
    if seconds < 7200:
        return f"{seconds / 60:.3g} minutes"
    if seconds < 172800:
        return f"{seconds / 3600:.3g} hours"
    return f"{seconds / 86400:.3g} days"


def bytes_in_words(count):
    """A number of bytes in the unit that suits it."""
    for size, unit in ((1e12, "TB"), (1e9, "GB"), (1e6, "MB"), (1e3, "KB")):
        if count >= size:
            return f"{count / size:.3g} {unit}"
    return f"{count:.0f} bytes"


def random_system(n):
    """A system of n equations whose matrix elimination handles without swapping rows: random numbers, with n added
    to the diagonal."""
    A = rng.random((n, n)) + n * np.eye(n)
    b = rng.random(n)
    return A, b


def uses(function, names):
    """True when the code of the function (not its comments or docstring) mentions one of the names."""
    try:
        tree = ast.parse(textwrap.dedent(inspect.getsource(function)))
    except (OSError, TypeError, SyntaxError):
        return False
    found = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.Name):
            found.add(node.id)
        elif isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, (ast.Import, ast.ImportFrom)):
            found.update(alias.name.split(".")[0] for alias in node.names)
            if isinstance(node, ast.ImportFrom) and node.module:
                found.add(node.module.split(".")[0])
    return bool(found & set(names))


print("Setup is done.")

## <font color="#1967d2"><b>Part 1: Elimination, with loops</b></font>

The method of the first notebook, written for n equations, is **Gaussian elimination**, and it has two halves. **Forward elimination** goes through the unknowns in order: for unknown k, it subtracts a multiple of equation k from every equation below it, so that the unknown disappears from them. When it is done, the last equation holds one unknown, the one before it two, and so on: the matrix is **upper triangular**. **Back substitution** then solves the last equation for the last unknown, puts it into the equation above to find the one before, and goes up to the first.

The next cell is forward elimination with three loops, written like the layer of neurons of meeting 12 so that its operations can be counted: the innermost line, `A[i, j] -= factor * A[k, j]`, runs for every unknown k, every row i below k and every column j from k on. It does not swap rows, which a real solver does when the number it divides by is small; the systems of this notebook are made so that it never has to.

In [ ]:
def eliminate(A, b):
    """Forward elimination: a copy of A made upper triangular, and b changed with it, so that the solutions stay the same."""
    A, b = A.astype(float).copy(), b.astype(float).copy()
    n = len(b)
    for k in range(n):                       # the unknown to remove from the rows below row k
        for i in range(k + 1, n):            # every row below row k
            factor = A[i, k] / A[k, k]       # the multiple of row k that removes the unknown from row i
            for j in range(k, n):            # every column from k on; the columns before k are already 0
                A[i, j] -= factor * A[k, j]
            b[i] -= factor * b[k]
    return A, b


A, b = random_system(4)
U, c = eliminate(A, b)
np.set_printoptions(precision=3, suppress=True)
print("the matrix after elimination, upper triangular:")
print(U)
print("the right-hand side, changed with it:", c)

<font color="#c00000" size="5"><b>Your turn.</b></font> Complete `back_substitute` in the next cell: it receives the upper triangular matrix `U` and the right-hand side `c`, and finds the unknowns from the last to the first. Row i of `U` reads `U[i, i] * x[i] + U[i, i + 1] * x[i + 1] + ... = c[i]`, and when row i is reached, the unknowns after i are already known: `U[i, i + 1:] @ x[i + 1:]` is their part of the sum. Replace the `...` with the line that computes `x[i]`. Then run the check.

In [ ]:
def back_substitute(U, c):
    """The solution of U x = c for an upper triangular U, from the last unknown to the first."""
    n = len(c)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):           # the rows from the last to the first
        x[i] = ...                           # Your turn: c[i], minus the part of the unknowns already found, divided by U[i, i]
    return x

In [ ]:
A, b = random_system(6)
U, c = eliminate(A, b)
try:
    x = back_substitute(U, c)
except TypeError:
    raise AssertionError("Replace the ... in back_substitute, and run that cell again.") from None
expected = np.linalg.solve(A, b)
assert np.shape(x) == expected.shape, "back_substitute must return one number for each unknown."
if not np.allclose(x, expected):
    n = len(c)
    no_division, wrong_side, plus = np.zeros(n), np.zeros(n), np.zeros(n)
    for i in range(n - 1, -1, -1):
        no_division[i] = c[i] - U[i, i + 1:] @ no_division[i + 1:]
        wrong_side[i] = (c[i] - U[i, :i] @ wrong_side[:i]) / U[i, i]
        plus[i] = (c[i] + U[i, i + 1:] @ plus[i + 1:]) / U[i, i]
    if np.allclose(x, no_division):
        raise AssertionError("Each x[i] must be divided by U[i, i], the number that multiplies x[i] in row i.")
    if np.allclose(x, wrong_side):
        raise AssertionError("The unknowns already found when row i is reached are the ones after i: U[i, i + 1:] @ x[i + 1:], "
                             "not the ones before it.")
    if np.allclose(x, plus):
        raise AssertionError("The part of the unknowns already found is subtracted from c[i], not added: minus, not plus.")
    raise AssertionError("x does not solve the system: x[i] = (c[i] - U[i, i + 1:] @ x[i + 1:]) / U[i, i].")
print("Correct: back_substitute finds the solution; eliminate and back_substitute together solve the system.")

In [ ]:
loop_sizes = [50, 100, 200]
loop_times = []
print(f"{'n':>6s}{'passes, about n^3 / 3':>24s}{'seconds':>10s}{'ratio to the first':>21s}{'nanoseconds per pass':>23s}")
for n in loop_sizes:
    A, b = random_system(n)
    start = time.perf_counter()
    x = back_substitute(*eliminate(A, b))
    loop_times.append(time.perf_counter() - start)
    assert np.allclose(x, np.linalg.solve(A, b)), "the loops and np.linalg.solve disagree"
    passes = n ** 3 // 3
    print(f"{n:6d}{passes:24,d}{loop_times[-1]:10.3f}{loop_times[-1] / loop_times[0]:21.1f}{loop_times[-1] / passes * 1e9:23.0f}")

**How to read the result.** The innermost line runs about n³/3 times (the sum of (n - k)² over k), two arithmetic operations each: the count of elimination is **O(n³)**, like the product of two matrices of size n, and doubling n multiplies the time by about 8. The loops of Python take a few hundred nanoseconds for each pass; a system of 4,000 unknowns, 21 billion passes, would take them hours. Nobody solves a system with the loops of Python: the next part hands the same work to compiled code.

## <font color="#1967d2"><b>Part 2: Three ways to solve</b></font>

The same system solved three ways: the loops of Part 1, the **inverse** of the matrix, `np.linalg.inv(A) @ b`, and `np.linalg.solve(A, b)`. The inverse of `A` is the matrix `A_inv` for which `A_inv @ A` is the identity, so that `x = A_inv @ b`; computing it is elimination applied to n right-hand sides at once, about 2n³ operations, three times the elimination of one system, and then a product. `np.linalg.solve` does what Part 1 did, in compiled code, with rows swapped where needed, and nothing more. The next cell times the two at sizes from 500 to 4,000; the loops were timed in Part 1 at the sizes they can manage. It takes up to a minute.

In [ ]:
sizes = [500, 1000, 2000, 4000]
seconds = {"inverse": [], "solve": []}
for n in sizes:
    A, b = random_system(n)
    seconds["inverse"].append(best_time(lambda: np.linalg.inv(A) @ b, repeats=2))
    seconds["solve"].append(best_time(lambda: np.linalg.solve(A, b), repeats=2))

print(f"{'n':>6s}{'np.linalg.inv(A) @ b':>22s}{'np.linalg.solve(A, b)':>24s}{'inverse / solve':>18s}")
for i, n in enumerate(sizes):
    print(f"{n:6d}{seconds['inverse'][i]:22.4f}{seconds['solve'][i]:24.4f}{seconds['inverse'][i] / seconds['solve'][i]:18.1f}")

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.plot(loop_sizes, loop_times, marker="^", color="#888888", label="Python: three loops")
ax.plot(sizes, seconds["inverse"], marker="o", color="#eb6834", label="np.linalg.inv(A) @ b")
ax.plot(sizes, seconds["solve"], marker="s", color="#2a78d6", label="np.linalg.solve(A, b)")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xticks(loop_sizes + sizes)
ax.set_xticklabels([str(n) for n in loop_sizes + sizes])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, position: f"{value:g}"))
ax.set_xlabel("n: the number of equations and of unknowns")
ax.set_ylabel("seconds for one solution")
ax.set_title("Three ways to solve A x = b (each step up the axis is 10 times longer)")
ax.grid(True, which="major", color="#e1e0d9", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

print("Line for the discussion board:")
print(f"Solving n = 4,000 equations: the inverse {seconds['inverse'][-1]:.3f} s, solve {seconds['solve'][-1]:.3f} s; "
      f"the loops of Python at n = 200: {loop_times[-1]:.2f} s, so about {duration(loop_times[-1] * 8000)} at n = 4,000.")

<font color="#c00000" size="5"><b>Your turn.</b></font> Read two numbers from the table, and type them in the next cell in place of the `...`:

1. `INV_OVER_SOLVE`: at n = 4,000, the seconds of the inverse divided by the seconds of `solve`.
2. `SOLVE_GROWTH`: by what factor the time of `solve` grows from n = 2,000 to n = 4,000.

In [ ]:
INV_OVER_SOLVE = ...      # Your turn: at n = 4,000, the seconds of the inverse divided by the seconds of solve
SOLVE_GROWTH = ...        # Your turn: the seconds of solve at n = 4,000 divided by its seconds at n = 2,000

In [ ]:
for name, value in (("INV_OVER_SOLVE", INV_OVER_SOLVE), ("SOLVE_GROWTH", SOLVE_GROWTH)):
    assert value is not ..., f"Replace the ... after {name} with a number from the table, and run that cell again."
    assert isinstance(value, (int, float)), f"Type {name} as a number, for example 2.5, not as text."
ratio = seconds["inverse"][-1] / seconds["solve"][-1]
growth = seconds["solve"][-1] / seconds["solve"][-2]
if abs(INV_OVER_SOLVE / (1 / ratio) - 1) < 0.1 and abs(INV_OVER_SOLVE / ratio - 1) >= 0.1:
    raise AssertionError("INV_OVER_SOLVE is the inverse's seconds divided by solve's, the last column of the row n = 4000: "
                         "larger than 1 when the inverse is slower.")
assert abs(INV_OVER_SOLVE / ratio - 1) < 0.1, (
    "INV_OVER_SOLVE does not agree with your table: in the row n = 4000, divide the seconds of the inverse by the "
    "seconds of solve (the last column holds that number).")
assert abs(SOLVE_GROWTH / growth - 1) < 0.1, (
    "SOLVE_GROWTH does not agree with your table: in the column of solve, divide the seconds in the row n = 4000 "
    "by the seconds in the row n = 2000.")
print(f"Correct: the inverse takes {ratio:.1f} times as long as solve at n = 4,000, and from 2,000 to 4,000 the time "
      f"of solve grows by a factor of {growth:.1f}.")

**How to read the result.** Both grow like n³, so the growth from 2,000 to 4,000 is near 8 for both, and the inverse costs more at every size: much more on a machine with two cores, less where many cores share the work, because the inverse's extra work divides among cores better. The reason to prefer `solve` is not only the time. It is also more accurate, as Part 3 measures, and it keeps a sparse matrix sparse, as Part 4 shows: the inverse of a matrix that is mostly zeros is usually full. If you find yourself writing `inv(A) @ b`, write `solve(A, b)`.

**The same O( ), another constant.** Answer in the next cell, in a few sentences: the inverse and `solve` are both O(n³), yet the inverse takes longer. Where does the difference come from, and what does the notation O( ) leave out here? Use the count of Part 1.

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

## <font color="#1967d2"><b>Part 3: Round-off, and the condition number</b></font>

A number in NumPy is stored in 8 bytes and keeps about 16 decimal digits; every arithmetic operation rounds its result to that. Elimination does n³/3 operations, each rounded, and the question is how much of the error reaches the solution. The next cell builds systems whose solution it knows, `b = A @ x_true`, for a matrix of 100 equations that are nearly the same, `np.ones((n, n)) + eps * np.eye(n)`: the nearly parallel lines of the first notebook, in 100 dimensions, with `eps` setting how nearly. It solves each with `solve` and with the inverse and prints the condition number, the **relative error** of each solution, the length of `x - x_true` divided by the length of `x_true`, and its **residual**, the length of `A @ x - b` divided by the length of `b`, which says how well the computed x satisfies the equations.

In [ ]:
n = 100
x_true = rng.random(n)
error = lambda x: np.linalg.norm(x - x_true) / np.linalg.norm(x_true)
print(f"{'eps':>8s}{'condition number':>18s}{'error of solve':>16s}{'error of inverse':>18s}{'residual of solve':>19s}{'residual of inverse':>21s}")
for eps in [1e-2, 1e-6, 1e-10, 1e-14]:
    A = np.ones((n, n)) + eps * np.eye(n)
    b = A @ x_true
    x_solve = np.linalg.solve(A, b)
    x_inverse = np.linalg.inv(A) @ b
    residual = lambda x: np.linalg.norm(A @ x - b) / np.linalg.norm(b)
    print(f"{eps:8.0e}{np.linalg.cond(A):18.1e}{error(x_solve):16.1e}{error(x_inverse):18.1e}"
          f"{residual(x_solve):19.1e}{residual(x_inverse):21.1e}")

**How to read the result.** Read the numbers as powers of ten. The condition number is about 100 / eps, and the error of `solve` is about the condition number times 10⁻¹⁶: with condition number 10⁸ the solution keeps about 8 of its 16 digits, with 10¹² about 4, and with 10¹⁶ none, the solution is off by about as much as it is large. The rule: **digits of the solution ≈ 16 - log₁₀(condition number)**. The inverse loses more. And the residual of `solve` is about 10⁻¹⁶ in every row: the computed x satisfies the equations as well as any x could, even when it is far from `x_true`. That is a property of a well-made solver, and it means that the residual cannot tell you that a solution is wrong; only the condition number can. Compute `np.linalg.cond(A)` before you trust a solution.

<font color="#c00000" size="5"><b>Your turn.</b></font> A system has condition number 10¹⁰. In the next cell, type how many digits of its solution you can expect to be right, `DIGITS_LEFT`. Then run the check.

In [ ]:
DIGITS_LEFT = ...      # Your turn: the digits of the solution that are right when the condition number is 10 to the 10

In [ ]:
assert DIGITS_LEFT is not ..., "Replace the ... after DIGITS_LEFT with a number, and run that cell again."
assert isinstance(DIGITS_LEFT, (int, float)), "Type DIGITS_LEFT as a number, not as text."
if DIGITS_LEFT == 10:
    raise AssertionError("10 is the number of digits lost, not left: a computer keeps about 16, and the condition number "
                         "costs up to log10 of itself.")
if DIGITS_LEFT >= 16:
    raise AssertionError("A computer keeps about 16 digits, and the condition number costs some of them: subtract.")
assert abs(DIGITS_LEFT - 6) <= 1, "Digits of the solution: about 16 minus log10 of the condition number."
print("Correct: about 6 digits, 16 minus 10.")

**A tiny residual, a wrong answer.** Answer in the next cell, in a few sentences: in the last row of the table, the residual of `solve` is about 10⁻¹⁶ and its error is about 1. How can an x satisfy the equations almost exactly and still be far from the true solution? Use the picture of nearly parallel lines.

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

## <font color="#1967d2"><b>Part 4: Matrices that are mostly zeros</b></font>

In a large system each equation usually involves a few of the unknowns, and the matrix is mostly zeros: **sparse**. The example of this notebook is a **grid**: n × n points, each tied to its four neighbours, up, down, left and right. One unknown for each point gives N = n² equations, each with five nonzero numbers, 4 on the diagonal and -1 for each neighbour, however large the grid. (Equations of this shape describe heat spreading over a plate, the pressure in a flow, and the walk on a graph of the first notebook.) The dense matrix of a 100 × 100 grid has 10,000 rows and 10,000 columns, 100 million numbers, 800 megabytes, and 50,000 of them are not zero.

A sparse matrix stores only its nonzeros, in one of three **formats**:

- **COO**, coordinates: three arrays of the same length, the row, the column and the value of each nonzero. The easiest to build.
- **CSR**, compressed sparse rows: the values and their columns, row by row, in two arrays, and a third array, `indptr`, of N + 1 numbers that says where each row begins in them. The best for products `A @ v`, which read the matrix by rows.
- **CSC**, compressed sparse columns: the same by columns. What most sparse solvers want.

The next cell builds the matrix of a 3 × 3 grid, N = 9, in COO, converts it to CSR and to CSC, and prints the arrays of each beside the dense matrix.

In [ ]:
def grid_matrix(n):
    """The matrix of an n x n grid in COO format: 4 on the diagonal, -1 for each of the four neighbours of a point."""
    rows, cols, values = [], [], []
    for i in range(n):
        for j in range(n):
            k = i * n + j                            # the point in row i and column j of the grid is unknown k
            rows.append(k)
            cols.append(k)
            values.append(4.0)
            for di, dj in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                if 0 <= i + di < n and 0 <= j + dj < n:
                    rows.append(k)
                    cols.append((i + di) * n + j + dj)
                    values.append(-1.0)
    return sparse.coo_matrix((values, (rows, cols)), shape=(n * n, n * n))


A_coo = grid_matrix(3)
A_csr = A_coo.tocsr()
A_csc = A_coo.tocsc()
np.set_printoptions(linewidth=140)
print("the dense matrix of the 3 x 3 grid, 9 rows and 9 columns:")
print(A_coo.toarray().astype(int))
print(f"nonzeros: {A_coo.nnz} of {A_coo.shape[0] * A_coo.shape[1]} numbers")
print("COO  row    ", A_coo.row)
print("     col    ", A_coo.col)
print("     data   ", A_coo.data.astype(int))
print("CSR  indptr ", A_csr.indptr, "  where each row begins in the two arrays below; the last entry is the number of nonzeros")
print("     indices", A_csr.indices, "  the column of each nonzero, row by row")
print("     data   ", A_csr.data.astype(int))
print("CSC  indptr ", A_csc.indptr, "  where each column begins")
print("     indices", A_csc.indices, "  the row of each nonzero, column by column")
print("     data   ", A_csc.data.astype(int))

**How to read the arrays.** COO lists the 33 nonzeros as 33 triples. CSR keeps the same values and their columns, row by row; `indptr` says that row 0 occupies positions 0 to 2 of the two arrays, row 1 positions 3 to 6, and so on, and its last entry, 33, is the number of nonzeros: the rows themselves are never stored. CSC does the same by columns; for this matrix, which equals its own transpose, the arrays come out the same. Dense storage costs 8 bytes for every number of the matrix; CSR costs 12 bytes for each nonzero, 8 for the value and 4 for its column, plus 4 for each row.

<font color="#c00000" size="5"><b>Your turn.</b></font> Ask an AI assistant (Gemini in Colab, ChatGPT, Claude, or another) to write the function `dense_to_csr`, which converts a dense NumPy array into the three arrays of the CSR format without scipy. State all of these requirements in your prompt:

- the function is named `dense_to_csr` and takes one argument, `A`, a two-dimensional NumPy array;
- it returns three NumPy arrays, `data`, `indices` and `indptr`: `data` holds the nonzero values of `A` row by row, left to right within a row; `indices` holds the column of each of them; `indptr` has one more entry than `A` has rows, `indptr[i]` is the position in `data` where row i begins, and the last entry is the number of nonzeros;
- `indices` and `indptr` hold whole numbers;
- it uses NumPy only, no scipy.

Paste your prompt into the next cell, and the function into the code cell after it. Then run the check, which compares your arrays with scipy's.

<font color="#c00000" size="5"><b>Your prompt.</b></font> Double-click this cell and replace this line with the prompt you gave to the AI assistant. If the first function did not pass the check, add one sentence on what you changed.

In [ ]:
# Your turn: paste the function dense_to_csr here, then run this cell.

In [ ]:
assert "dense_to_csr" in globals(), "Paste the function dense_to_csr into the cell above and run that cell."
assert not uses(dense_to_csr, ("scipy", "sparse", "csr_matrix", "csr_array")), (
    "dense_to_csr must build the three arrays itself, without scipy.")
T = rng.random((5, 7))
T[T < 0.6] = 0                                   # a small matrix with zeros in it
result = dense_to_csr(T)
assert isinstance(result, (tuple, list)) and len(result) == 3, "dense_to_csr must return three arrays: data, indices, indptr."
data, indices, indptr = (np.asarray(r) for r in result)
reference = sparse.csr_matrix(T)
assert len(indptr) == T.shape[0] + 1, (
    f"indptr has {len(indptr)} entries; it must have one more than the rows of A, {T.shape[0] + 1}: where each row "
    "begins, and the number of nonzeros at the end.")
assert np.issubdtype(indices.dtype, np.integer) and np.issubdtype(indptr.dtype, np.integer), (
    "indices and indptr must hold whole numbers.")
assert np.array_equal(indptr, reference.indptr), (
    f"indptr is {indptr.tolist()}; scipy's is {reference.indptr.tolist()}. indptr[i] is where row i begins in data; "
    "the last entry is the number of nonzeros.")
assert np.array_equal(indices, reference.indices), (
    "indices does not list the columns of the nonzeros row by row, left to right within each row. Is the matrix "
    "scanned by rows, not by columns?")
assert np.allclose(data, reference.data), "data must hold the nonzero values in the order of indices."
print("Correct: dense_to_csr gives the same three arrays as scipy.sparse.csr_matrix.")

In [ ]:
print(f"{'grid':>8s}{'N = unknowns':>14s}{'nonzeros':>12s}{'dense, 8 bytes a number':>26s}{'CSR, 12 bytes a nonzero':>26s}")
for n in [10, 30, 100, 300, 1000, 3000]:
    N = n * n
    nonzeros = N + 4 * n * (n - 1)               # the diagonal, and two entries for each edge of the grid
    print(f"{n:5d} x {n:<3d}{N:14,d}{nonzeros:12,d}{bytes_in_words(8 * N * N):>26s}{bytes_in_words(12 * nonzeros + 4 * (N + 1)):>26s}")
print("check on the 100 x 100 grid, built:", grid_matrix(100).nnz, "nonzeros")

**How to read the table.** The dense matrix grows with N², the sparse one with N: for the 1,000 × 1,000 grid, one million unknowns, the dense matrix would need 8 terabytes and the sparse one 70 megabytes. Every large system in practice is stored sparse, and the question of the next part is whether it can be solved that way too.

## <font color="#1967d2"><b>Part 5: Solving, dense and sparse</b></font>

`scipy.sparse.linalg.spsolve` solves a system whose matrix is sparse, by elimination that skips the zeros; `np.linalg.solve` needs the dense array. The next cell times both on grids from n = 20 to n = 80 (N = 400 to 6,400 unknowns), and the sparse solver alone up to n = 320 (N = 102,400), for which the dense matrix would need 84 gigabytes. It takes up to a minute.

In [ ]:
grid_sizes = [20, 40, 60, 80, 160, 320]
dense_sizes = [20, 40, 60, 80]                   # the dense matrix of the 80 x 80 grid is 328 MB; the next would be 5 GB
unknowns, dense_seconds_solve, sparse_seconds_solve = [], [], []
for n in grid_sizes:
    A_sparse = grid_matrix(n).tocsc()
    N = n * n
    b = rng.random(N)
    unknowns.append(N)
    sparse_seconds_solve.append(best_time(lambda: sparse_linalg.spsolve(A_sparse, b), repeats=2))
    if n in dense_sizes:
        A_dense = A_sparse.toarray()
        dense_seconds_solve.append(best_time(lambda: np.linalg.solve(A_dense, b), repeats=2))
        del A_dense

print(f"{'grid':>10s}{'N':>9s}{'nonzeros':>10s}{'np.linalg.solve, dense':>25s}{'spsolve, sparse':>18s}")
for i, n in enumerate(grid_sizes):
    dense_text = f"{dense_seconds_solve[i]:.4f} s" if i < len(dense_seconds_solve) else "too large to store"
    print(f"{n:5d} x {n:<3d}{unknowns[i]:9,d}{grid_matrix(n).nnz:10,d}{dense_text:>25s}{sparse_seconds_solve[i]:16.4f} s")

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.plot(unknowns[:len(dense_seconds_solve)], dense_seconds_solve, marker="o", color="#eb6834", label="np.linalg.solve, dense")
ax.plot(unknowns, sparse_seconds_solve, marker="s", color="#2a78d6", label="scipy.sparse.linalg.spsolve")
ax.set_xscale("log")
ax.set_yscale("log")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, position: f"{value:g}"))
ax.set_xlabel("N: the number of unknowns (each mark on the axis is 10 times the one before)")
ax.set_ylabel("seconds for one solution")
ax.set_title("The system of a grid, solved dense and sparse")
ax.grid(True, which="major", color="#e1e0d9", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

print("Line for the discussion board:")
print(f"The grid of 80 x 80, N = 6,400: dense {dense_seconds_solve[-1]:.3f} s, sparse {sparse_seconds_solve[3]:.4f} s; "
      f"the grid of 320 x 320, N = 102,400: sparse {sparse_seconds_solve[-1]:.3f} s, dense not possible.")

**How to read the result.** The dense solver's time grows like N³: four times the unknowns, sixty-four times the seconds, and at N = 25,600 it cannot even store the matrix. The sparse solver's time grows far more slowly and reaches sizes the dense one never will. It is not O(nnz), though: elimination creates new nonzeros as it goes, called **fill-in**, and how many depends on the order in which the unknowns are eliminated. That is the deeper look of the meeting.

## <font color="#1967d2"><b>Part 6: The cost of a product, as the matrix fills</b></font>

A product `A @ v` costs about N² operations for a dense matrix, one for each number of the matrix, zero or not. For a sparse matrix in CSR it costs about nnz, one for each nonzero: **O(nnz)**, the cost model of sparse computing. The next cell times both on random matrices with N = 4,000 whose **density**, the fraction of entries that are not zero, goes from one in a thousand to all of them. Where does the sparse product stop winning?

In [ ]:
N = 4000
v = rng.random(N)
A_dense = rng.random((N, N))
dense_seconds = best_time(lambda: A_dense @ v, repeats=5)
densities = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]
sparse_seconds, nonzeros = [], []
for density in densities:
    A_sparse = sparse.random(N, N, density=density, format="csr", random_state=1)
    nonzeros.append(A_sparse.nnz)
    sparse_seconds.append(best_time(lambda: A_sparse @ v, repeats=5))
del A_sparse

print(f"dense product, {N:,} x {N:,} numbers: {dense_seconds * 1e3:.2f} milliseconds, {dense_seconds / N ** 2 * 1e9:.2f} nanoseconds a number")
print(f"{'density':>9s}{'nonzeros':>12s}{'sparse product, ms':>21s}{'ns a nonzero':>14s}{'faster':>9s}")
for density, nnz, s in zip(densities, nonzeros, sparse_seconds):
    print(f"{density:9g}{nnz:12,d}{s * 1e3:21.2f}{s / nnz * 1e9:14.2f}{'sparse' if s < dense_seconds else 'dense':>9s}")

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.plot(densities, [s * 1e3 for s in sparse_seconds], marker="s", color="#2a78d6", label="sparse, CSR")
ax.axhline(dense_seconds * 1e3, color="#eb6834", label="dense")
ax.set_xscale("log")
ax.set_xlabel("density: the fraction of the entries that are not zero")
ax.set_ylabel("milliseconds for A @ v")
ax.set_title(f"A @ v for N = {N:,}: sparse against dense, as the matrix fills")
ax.grid(True, which="major", color="#e1e0d9", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

<font color="#c00000" size="5"><b>Your turn.</b></font> Read from the table the smallest density at which the dense product is faster than the sparse one, and type it in the next cell as `CROSSOVER_DENSITY` (or `None`, if the sparse product won at every density). Then run the check.

In [ ]:
CROSSOVER_DENSITY = ...      # Your turn: the smallest density of the table at which the dense product is faster, or None

In [ ]:
assert CROSSOVER_DENSITY is not ..., "Replace the ... after CROSSOVER_DENSITY, and run that cell again."
expected = next((d for d, s in zip(densities, sparse_seconds) if dense_seconds < s), None)
if CROSSOVER_DENSITY is not None:
    assert isinstance(CROSSOVER_DENSITY, (int, float)), "Type CROSSOVER_DENSITY as a number, such as 0.1, not as text."
    assert any(np.isclose(CROSSOVER_DENSITY, d) for d in densities), (
        "CROSSOVER_DENSITY must be one of the densities of the table, the first row in which the dense product is faster.")
assert (CROSSOVER_DENSITY is None and expected is None) or (
    CROSSOVER_DENSITY is not None and expected is not None and np.isclose(CROSSOVER_DENSITY, expected)), (
    "That is not the first density of your table at which the dense product is faster. Read the last column from the top.")
print("Correct:", "the sparse product won at every density of the table." if expected is None
      else f"from a density of {expected:g} on, the dense product is faster on this machine.")

**How to read the result.** The sparse product costs a few nanoseconds for each nonzero, several times the dense product's cost for each number, because each nonzero carries its column index, read from memory with it, and the numbers of `v` are fetched in no particular order: memory-bound work, in the words of meeting 12. So the sparse form wins while the density is below some tens of per cent, and loses above. The grid is nowhere near: its density is 5 / N, one entry in two thousand for N = 10,000.

**When sparse pays.** Answer in the next cell, in a few sentences: from Parts 4 to 6, when is a matrix worth storing sparse, and when not? Give the count of operations and the bytes.

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

<font color="#c00000" size="5"><b>Post now</b></font>, as a reply to your group's first post. One member posts for the group; begin with the names of all its members. The post contains:

- the chart of Part 2, the three ways to solve, with the line for the discussion board printed under it, and one sentence that says what it shows;
- the chart of Part 5, the grid solved dense and sparse, with its line, and one sentence;
- your `CROSSOVER_DENSITY` of Part 6, and the number of cores of your machine if you know it.

## Upload your notebooks

**Every student uploads their own copies of both notebooks of meeting 13, before the meeting begins.** They are graded on the cells marked <font color="#c00000"><b>Your thinking</b></font> and <font color="#c00000"><b>Your prompt</b></font>: four in the first notebook and four in this one. Check that each holds your own words and not the line that was there.

1. In each notebook, open the **File** menu, choose **Download**, and then **Download .ipynb**.
2. On Canvas, open the module of meeting 13, choose **Upload Your Class Work - Meeting 13**, and upload **both files** in the same submission.